In [1]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
import joblib
import pandas as pd
from datetime import date

# --- 1. Load Resources ---
# Load the saved model and feature list
try:
    MODEL = joblib.load('demand_model.joblib')
    MODEL_FEATURES = joblib.load('model_features.joblib')
except FileNotFoundError:
    print("FATAL ERROR: Model files (demand_model.joblib or model_features.joblib) not found.")
    exit()

# Static lists for dropdowns
PRODUCT_CATEGORIES = ['Freshy Coconut', 'Nut Milk', 'Juicy Juice']
PRODUCT_NAMES = ['Coconut Water', 'Kelapa Ijo', 'Choco Blast', 'Vanilla Shake', 'Tropical Vibes', 'Dynamic Green', 'Sun Kissed']
DAY_OF_WEEKS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# --- 2. Initialize App and Define Layout ---
app = dash.Dash(__name__)

app.layout = html.Div(style={'padding': '20px', 'font-family': 'Arial, sans-serif'}, children=[
    html.H1("IDJuicer Demand Forecast Predictor (Dash)", style={'textAlign': 'center'}),
    html.P("Enter the parameters below and click 'Predict' to get the forecast."),

    html.Div([
        # --- INPUT COLUMN 1: Date & Operational ---
        html.Div(style={'width': '48%', 'display': 'inline-block', 'padding': '10px'}, children=[
            html.H3("Date & Operational Parameters"),
            
            html.Div("Prediction Date:", style={'marginTop': '10px'}),
            dcc.DatePickerSingle(
                id='input-date',
                initial_visible_month=date.today(),
                date=date.today(),
                display_format='YYYY-MM-DD'
            ),
            
            html.Div("Is Weekend:", style={'marginTop': '10px'}),
            dcc.Dropdown(
                id='input-is_weekend',
                options=[{'label': 'No (0)', 'value': 0}, {'label': 'Yes (1)', 'value': 1}],
                value=0,
                clearable=False
            ),

            html.Div("Is Pay Day:", style={'marginTop': '10px'}),
            dcc.Dropdown(
                id='input-pay_day',
                options=[{'label': 'No (0)', 'value': 0}, {'label': 'Yes (1)', 'value': 1}],
                value=0,
                clearable=False
            ),
        ]),

        # --- INPUT COLUMN 2: Product & Pricing ---
        html.Div(style={'width': '48%', 'display': 'inline-block', 'padding': '10px'}, children=[
            html.H3("Product & Pricing"),

            html.Div("Product Category:", style={'marginTop': '10px'}),
            dcc.Dropdown(id='input-category', options=[{'label': i, 'value': i} for i in PRODUCT_CATEGORIES], value=PRODUCT_CATEGORIES[0], clearable=False),

            html.Div("Product Name:", style={'marginTop': '10px'}),
            dcc.Dropdown(id='input-product', options=[{'label': i, 'value': i} for i in PRODUCT_NAMES], value=PRODUCT_NAMES[0], clearable=False),

            html.Div("Unit Cost (IDR):", style={'marginTop': '10px'}),
            dcc.Input(id='input-unit_cost', type='number', value=19000, min=1, style={'width': '100%'}),

            html.Div("Selling Price (IDR):", style={'marginTop': '10px'}),
            dcc.Input(id='input-selling_price', type='number', value=40000, min=1, style={'width': '100%'}),

            html.Div("Discount Rate:", style={'marginTop': '10px'}),
            dcc.Slider(id='input-discount_rate', min=0, max=0.5, step=0.01, value=0.1, marks={i/10: f'{i/10}' for i in range(6)}),
        ]),
    ], style={'display': 'flex', 'justify-content': 'space-between', 'border': '1px solid #ccc', 'padding': '15px', 'borderRadius': '5px'}),
    
    html.Button('Get Demand Forecast', id='predict-button', n_clicks=0, style={'marginTop': '20px', 'padding': '10px 20px', 'backgroundColor': '#007bff', 'color': 'white', 'border': 'none', 'borderRadius': '5px'}),
    
    html.Hr(style={'marginTop': '30px'}),
    
    # --- OUTPUT AREA ---
    html.Div(id='output-prediction', style={'fontSize': '24px', 'fontWeight': 'bold', 'textAlign': 'center'})
])

# --- 3. Define Callback (Prediction Logic) ---
@app.callback(
    Output('output-prediction', 'children'),
    [Input('predict-button', 'n_clicks')],
    [State('input-date', 'date'),
     State('input-is_weekend', 'value'),
     State('input-pay_day', 'value'),
     State('input-category', 'value'),
     State('input-product', 'value'),
     State('input-unit_cost', 'value'),
     State('input-selling_price', 'value'),
     State('input-discount_rate', 'value')]
)
def update_output(n_clicks, date_str, is_weekend, pay_day, category, product, unit_cost, selling_price, discount_rate):
    # Only run prediction after the button has been clicked at least once
    if n_clicks == 0:
        return html.P("Click the 'Get Demand Forecast' button above.", style={'color': '#6c757d', 'fontSize': '18px'})

    # Handle potentially missing input values (e.g., if user clears a number field)
    if None in [unit_cost, selling_price, discount_rate]:
        return html.P("Please ensure all numerical fields have a value.", style={'color': 'red'})

    # A. Prepare Input Data
    input_date = pd.to_datetime(date_str).date()
    day_of_week = DAY_OF_WEEKS[input_date.weekday()]

    input_data = {
        'unit_cost': unit_cost,
        'discount_rate': discount_rate,
        'Selling_Price': selling_price,
        'is_weekend': is_weekend,
        'pay_day': pay_day,
        'Day_of_Week': day_of_week,
        'product_category': category,
        'product_name': product
    }
    input_df = pd.DataFrame(input_data, index=[0])

    # B. Feature Engineering
    input_df['Month'] = input_date.month
    input_df['Day_of_Year'] = input_date.timetuple().tm_yday
    
    # C. One-Hot Encoding and Alignment (CRUCIAL STEP)
    categorical_cols_to_encode = ['Day_of_Week', 'product_category', 'product_name']
    input_df = pd.get_dummies(input_df, columns=categorical_cols_to_encode, drop_first=True)

    # Align columns to match the training data (handle missing/extra features)
    final_input = pd.DataFrame(0, index=input_df.index, columns=MODEL_FEATURES)
    for col in final_input.columns:
        if col in input_df.columns:
            final_input[col] = input_df[col]

    # D. Make Prediction
    try:
        prediction = MODEL.predict(final_input)
        forecast_units = max(0, round(float(prediction[0])))
        
        # E. Return the formatted output
        return [
            html.P(f"Predicted Demand for {product}:", style={'color': '#007bff'}),
            html.P(f"{forecast_units:,.0f} Units", style={'fontSize': '48px', 'color': '#28a745'}),
            html.P("Model validated with an average error (MAE) of 3.63 units.", style={'fontSize': '14px', 'color': '#6c757d'})
        ]

    except Exception as e:
        return html.P(f"Prediction Failed. Error: {e}", style={'color': 'red'})

# --- 4. Run the App ---
if __name__ == '__main__':
    # Set debug=True for development, change to False for production
    app.run(debug=True)